# Denmark Electricity Forecasting Dataset Pipeline

**Mục tiêu:** Tạo dataset giờ-theo-giờ cho DK1 & DK2 (2022–2024) với các cột:
`timestamp_utc | zone | spot_price_eur | imbalance | mfrr_activated | consumption_mwh | wind_speed | radia_glob | temp_dry`

**Nguồn dữ liệu:**
- [Energi Data Service](https://www.energidataservice.dk/) — Giá điện, cân bằng lưới, tiêu thụ
- [DMI Open Data API](https://opendataapi.dmi.dk) — Dữ liệu thời tiết

---
✅ **Không cần API key DMI:** Kể từ 2/12/2025, tất cả endpoints tại `opendataapi.dmi.dk` đều mở hoàn toàn — không cần đăng ký hay xác thực. Chỉ cần tuân thủ [fair use policy](https://www.dmi.dk/friedata/dokumentation/terms-of-use).

## 0. Cài đặt thư viện

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install requests pandas numpy pytz tqdm --quiet

## 1. Cấu hình

In [ ]:
import requests
import pandas as pd
import numpy as np
import pytz
import json
import time
import os
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CẤU HÌNH
# ============================================================

# Phạm vi thời gian
START_DATE = "2022-01-01"
END_DATE   = "2024-12-31"

# Thư mục lưu dữ liệu thô (cache)
RAW_DIR = "./raw_data"
os.makedirs(RAW_DIR, exist_ok=True)

# Múi giờ Đan Mạch
TZ_DK = pytz.timezone("Europe/Copenhagen")
TZ_UTC = pytz.UTC

print("✅ Cấu hình xong.")
print(f"   Phạm vi: {START_DATE} → {END_DATE}")
print(f"   Thư mục cache: {os.path.abspath(RAW_DIR)}")

## 2. Trạm thời tiết DMI theo vùng

Danh sách trạm DMI được phân vùng DK1 (Jutland + Funen) và DK2 (Zealand + Bornholm).
Tham khảo: https://opendataapi.dmi.dk/stations

In [ ]:
# Trạm DMI — stationId theo vùng địa lý
# Jutland + Funen = DK1
STATIONS_DK1 = [
    "06030",  # Skagen
    "06041",  # Frederikshavn
    "06060",  # Aalborg
    "06079",  # Thyborøn
    "06081",  # Hvide Sande
    "06096",  # Billund
    "06102",  # Aarhus
    "06118",  # Odense
    "06156",  # Esbjerg
    "06180",  # Kolding
]

# Zealand + Bornholm = DK2
STATIONS_DK2 = [
    "06170",  # Roskilde
    "06180",  # Sjællands Odde (nếu có)
    "06183",  # Copenhagen / Kastrup
    "06190",  # Røsnæs
    "06193",  # Tystofte
    "06194",  # Stevns
    "06197",  # Gedser
    "06280",  # Bornholm / Rønne
]

# Biến thời tiết cần lấy từ DMI
DMI_PARAMETERS = ["wind_speed", "radia_glob", "temp_dry"]

print(f"DK1: {len(STATIONS_DK1)} trạm | DK2: {len(STATIONS_DK2)} trạm")

## 3. Helper functions

In [ ]:
def daterange_chunks(start: str, end: str, chunk_days: int = 90):
    """Chia phạm vi ngày thành các chunk nhỏ để tránh timeout API."""
    s = datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end, "%Y-%m-%d")
    cur = s
    while cur <= e:
        nxt = min(cur + timedelta(days=chunk_days - 1), e)
        yield cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")
        cur = nxt + timedelta(days=1)


def to_utc(series, tz_local=None):
    """
    Chuyển đổi chuỗi datetime từ múi giờ địa phương sang UTC.
    Xử lý an toàn DST: ambiguous='infer', nonexistent='shift_forward'
    """
    if tz_local is None:
        tz_local = TZ_DK
    import pandas as _pd
    if series.dt.tz is None:
        series = series.dt.tz_localize(tz_local,
                                        ambiguous='infer',
                                        nonexistent='shift_forward')
    return series.dt.tz_convert(TZ_UTC)


def eds_fetch(dataset: str, start: str, end: str,
              columns: list = None, filters: dict = None,
              limit: int = 100_000) -> 'pd.DataFrame':
    """
    Lấy dữ liệu từ Energi Data Service REST API.
    Docs: https://api.energidataservice.dk/

    Lưu ý quan trọng về EDS API:
      - start/end phải là ISO datetime: '2022-01-01T00:00' (không phải chỉ ngày)
      - sort: chỉ tên cột, KHÔNG dùng 'ASC'/'DESC' trong cùng string
      - filter: JSON compact không có space  (separators=(',',':'))
      - limit thực tế ~100_000 (250_000 hay timeout)
    """
    BASE = "https://api.energidataservice.dk/dataset"

    def _to_eds_dt(d: str) -> str:
        """Chuyển 'YYYY-MM-DD' → 'YYYY-MM-DDTHH:MM' nếu chưa có time component."""
        return d if 'T' in d else f"{d}T00:00"

    params = {
        "start":  _to_eds_dt(start),
        "end":    _to_eds_dt(end),
        "limit":  limit,
        "offset": 0,
        "sort":   "HourDK",   # EDS: chỉ tên cột; không dùng 'HourDK ASC'
    }
    if columns:
        params["columns"] = ",".join(columns)
    if filters:
        # QUAN TRỌNG: compact JSON — không có space sau ':' hay ','
        # json.dumps mặc định thêm space → EDS trả 400
        params["filter"] = json.dumps(filters, separators=(',', ':'))

    all_records = []
    while True:
        r = requests.get(f"{BASE}/{dataset}", params=params, timeout=120)
        if not r.ok:
            print(f"  ❌ HTTP {r.status_code} — URL: {r.url}")
            print(f"  Response: {r.text[:500]}")
        r.raise_for_status()
        data = r.json()
        records = data.get("records", [])
        all_records.extend(records)
        total = data.get("total", 0)
        if len(all_records) >= total or not records:
            break
        params["offset"] += limit
        time.sleep(0.5)  # rate-limit courtesy

    return pd.DataFrame(all_records)


def forward_fill_with_flag(df, cols: list, max_gap: int = 2):
    """
    - Gap <= max_gap giờ: forward-fill.
    - Gap > max_gap giờ: giữ NaN và thêm cột boolean *_gap_flag = True.
    """
    for col in cols:
        if col not in df.columns:
            continue
        flag_col = f"{col}_gap_flag"
        is_null = df[col].isna()

        # Đếm run-length của các NaN liên tiếp
        run_id = (~is_null).cumsum()
        run_len = is_null.groupby(run_id).transform('sum')

        # Flag gap dài
        df[flag_col] = is_null & (run_len > max_gap)

        # Forward-fill chỉ những gap ngắn
        short_gap = is_null & (run_len <= max_gap)
        df.loc[short_gap, col] = np.nan  # đảm bảo cờ đúng
        df[col] = df[col].ffill()
        # Khôi phục NaN cho gap dài
        df.loc[df[flag_col], col] = np.nan

    return df


print("✅ Helper functions đã sẵn sàng.")
print("   eds_fetch fixes: ISO datetime, compact JSON filter, sort without ASC/DESC")

## 4. Download: Giá điện ngày hôm trước (DayAheadPrices)

In [ ]:
CACHE_PRICES = os.path.join(RAW_DIR, "day_ahead_prices.parquet")

if os.path.exists(CACHE_PRICES):
    print("📂 Đọc từ cache...")
    df_prices = pd.read_parquet(CACHE_PRICES)
else:
    print("🌐 Đang tải DayAheadPrices từ Energi Data Service...")
    chunks = []
    for s, e in tqdm(list(daterange_chunks(START_DATE, END_DATE, 180))):
        chunk = eds_fetch(
            dataset  = "DayAheadPrices",
            start    = s,
            end      = e,
            columns  = ["HourDK", "PriceArea", "SpotPriceEUR"],
            filters  = {"PriceArea": ["DK1", "DK2"]}
        )
        chunks.append(chunk)
        time.sleep(0.3)

    df_prices = pd.concat(chunks, ignore_index=True)
    df_prices.to_parquet(CACHE_PRICES)
    print(f"   → {len(df_prices):,} hàng")

# Làm sạch
df_prices["HourDK"] = pd.to_datetime(df_prices["HourDK"])
df_prices["timestamp_utc"] = to_utc(df_prices["HourDK"])
df_prices = df_prices.rename(columns={
    "PriceArea":   "zone",
    "SpotPriceEUR": "spot_price_eur"
})[["timestamp_utc", "zone", "spot_price_eur"]]

df_prices = df_prices.drop_duplicates(subset=["timestamp_utc", "zone"])
print(f"✅ Giá điện: {len(df_prices):,} hàng | {df_prices['zone'].unique()}")
df_prices.head(3)

## 5. Download: Imbalance & mFRR (Realtime Electricity Market Data)

> ⚠️ Dataset cũ "Power System Right Now" đã xóa cột Imbalance/mFRR từ tháng 6/2026.
> Bắt buộc dùng dataset **RealtimeElectricityMarket** (hoặc tên mới nhất trên EDS).

In [ ]:
CACHE_RT = os.path.join(RAW_DIR, "realtime_market.parquet")

if os.path.exists(CACHE_RT):
    print("📂 Đọc từ cache...")
    df_rt = pd.read_parquet(CACHE_RT)
else:
    print("🌐 Đang tải Realtime Electricity Market Data...")
    # Tên dataset mới nhất — kiểm tra tại:
    # https://api.energidataservice.dk/dataset?filter={"datasetName":"Realtime"}
    DATASET_RT = "RealtimeElectricityMarket"

    chunks = []
    for s, e in tqdm(list(daterange_chunks(START_DATE, END_DATE, 90))):
        chunk = eds_fetch(
            dataset = DATASET_RT,
            start   = s,
            end     = e,
        )
        chunks.append(chunk)
        time.sleep(0.3)

    df_rt = pd.concat(chunks, ignore_index=True)
    df_rt.to_parquet(CACHE_RT)
    print(f"   → {len(df_rt):,} hàng")
    print(f"   Columns: {list(df_rt.columns)}")

print("\nCột hiện có:")
print(df_rt.columns.tolist())

In [ ]:
# ============================================================
# ĐIỀU CHỈNH TÊN CỘT nếu cần — chạy cell này sau khi xem columns ở trên
# ============================================================

# Mapping tên cột thực tế → tên chuẩn
# Kiểm tra df_rt.columns và cập nhật dict này cho đúng
IMBALANCE_COL_MAP = {
    # Ví dụ tên cột từ EDS (có thể thay đổi):
    "ImbalanceDK1": {"zone": "DK1", "field": "imbalance"},
    "ImbalanceDK2": {"zone": "DK2", "field": "imbalance"},
    "mFRRActivatedDK1": {"zone": "DK1", "field": "mfrr_activated"},
    "mFRRActivatedDK2": {"zone": "DK2", "field": "mfrr_activated"},
}

# Tên cột timestamp trong dataset RT
RT_TIME_COL = "HourDK"  # Cập nhật nếu khác

# Tái cấu trúc từ wide → long format
def reshape_rt(df_wide: pd.DataFrame,
               time_col: str,
               col_map: dict) -> pd.DataFrame:
    df_wide[time_col] = pd.to_datetime(df_wide[time_col])
    df_wide["timestamp_utc"] = to_utc(df_wide[time_col])

    records = []
    for zone in ["DK1", "DK2"]:
        imb_cols  = [c for c, v in col_map.items() if v["zone"] == zone and v["field"] == "imbalance"]
        mfrr_cols = [c for c, v in col_map.items() if v["zone"] == zone and v["field"] == "mfrr_activated"]

        imb_col  = imb_cols[0]  if imb_cols  else None
        mfrr_col = mfrr_cols[0] if mfrr_cols else None

        tmp = df_wide[["timestamp_utc"]].copy()
        tmp["zone"] = zone
        tmp["imbalance"]      = df_wide[imb_col].values  if imb_col  and imb_col  in df_wide.columns else np.nan
        tmp["mfrr_activated"] = df_wide[mfrr_col].values if mfrr_col and mfrr_col in df_wide.columns else np.nan
        records.append(tmp)

    return pd.concat(records, ignore_index=True)


df_balance = reshape_rt(df_rt, RT_TIME_COL, IMBALANCE_COL_MAP)
df_balance = df_balance.drop_duplicates(subset=["timestamp_utc", "zone"])
print(f"✅ Balance data: {len(df_balance):,} hàng")
df_balance.head(3)

## 6. Download: Tiêu thụ điện (ConsumptionDK3619CodeHour)

In [ ]:
CACHE_CONS = os.path.join(RAW_DIR, "consumption.parquet")

if os.path.exists(CACHE_CONS):
    print("📂 Đọc từ cache...")
    df_cons_raw = pd.read_parquet(CACHE_CONS)
else:
    print("🌐 Đang tải ConsumptionDK3619CodeHour...")
    chunks = []
    for s, e in tqdm(list(daterange_chunks(START_DATE, END_DATE, 90))):
        chunk = eds_fetch(
            dataset = "ConsumptionDK3619CodeHour",
            start   = s,
            end     = e,
            columns = ["HourDK", "PriceArea", "ConsumptionkWh"],
            filters = {"PriceArea": ["DK1", "DK2"]}
        )
        chunks.append(chunk)
        time.sleep(0.3)

    df_cons_raw = pd.concat(chunks, ignore_index=True)
    df_cons_raw.to_parquet(CACHE_CONS)
    print(f"   → {len(df_cons_raw):,} hàng")

# Aggregate: tổng theo giờ và vùng; kWh → MWh
df_cons_raw["HourDK"] = pd.to_datetime(df_cons_raw["HourDK"])
df_cons_raw["timestamp_utc"] = to_utc(df_cons_raw["HourDK"])

# Xử lý trường hợp tên cột tiêu thụ khác nhau
cons_val_col = "ConsumptionkWh" if "ConsumptionkWh" in df_cons_raw.columns else "Consumption_MWh"
df_cons_raw[cons_val_col] = pd.to_numeric(df_cons_raw[cons_val_col], errors="coerce")

df_cons = (
    df_cons_raw
    .groupby(["timestamp_utc", "PriceArea"], as_index=False)[cons_val_col]
    .sum()
    .rename(columns={"PriceArea": "zone", cons_val_col: "consumption_mwh"})
)

# kWh → MWh nếu cần
if "kWh" in cons_val_col:
    df_cons["consumption_mwh"] = df_cons["consumption_mwh"] / 1000

df_cons = df_cons.drop_duplicates(subset=["timestamp_utc", "zone"])
print(f"✅ Tiêu thụ: {len(df_cons):,} hàng")
df_cons.head(3)

## 7. Download: Dữ liệu thời tiết DMI

**API mới:** `https://opendataapi.dmi.dk` (API cũ đã ngừng hoạt động năm 2026)

In [ ]:
DMI_BASE = "https://opendataapi.dmi.dk/v2/metObs/collections/observation/items"

def dmi_fetch_station(station_id: str, parameter: str,
                      start: str, end: str) -> pd.DataFrame:
    """
    Lấy quan trắc theo giờ từ 1 trạm DMI cho 1 biến.
    API: https://opendataapi.dmi.dk (OGC Features API)
    Không cần API key kể từ 2/12/2025.
    """
    # DMI dùng ISO 8601 UTC
    start_iso = f"{start}T00:00:00Z"
    end_iso   = f"{end}T23:59:59Z"

    all_features = []
    params = {
        "stationId":   station_id,
        "parameterId": parameter,
        "datetime":    f"{start_iso}/{end_iso}",
        "timeResolution": "hour",
        "limit":       10_000,
        "offset":      0,
    }

    while True:
        r = requests.get(DMI_BASE, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        features = data.get("features", [])
        all_features.extend(features)
        if len(features) < params["limit"]:
            break
        params["offset"] += params["limit"]
        time.sleep(0.2)

    if not all_features:
        return pd.DataFrame(columns=["timestamp_utc", "station_id", parameter])

    rows = []
    for f in all_features:
        props = f["properties"]
        rows.append({
            "timestamp_utc": pd.to_datetime(props.get("observed")).tz_convert(TZ_UTC),
            "station_id":    station_id,
            parameter:       props.get("value"),
        })
    return pd.DataFrame(rows)


def fetch_dmi_zone(stations: list, parameter: str,
                   start: str, end: str,
                   chunk_days: int = 30) -> pd.DataFrame:
    """Lấy dữ liệu từ tất cả trạm trong một vùng và tính trung bình theo giờ."""
    all_dfs = []
    for station in tqdm(stations, desc=f"{parameter}", leave=False):
        station_dfs = []
        for s, e in daterange_chunks(start, end, chunk_days):
            try:
                df_s = dmi_fetch_station(station, parameter, s, e)
                station_dfs.append(df_s)
            except Exception as ex:
                print(f"   ⚠️ Lỗi trạm {station} [{s}→{e}]: {ex}")
            time.sleep(0.1)
        if station_dfs:
            all_dfs.append(pd.concat(station_dfs, ignore_index=True))

    if not all_dfs:
        return pd.DataFrame(columns=["timestamp_utc", parameter])

    combined = pd.concat(all_dfs, ignore_index=True)
    combined[parameter] = pd.to_numeric(combined[parameter], errors="coerce")

    # Resample sang giờ tròn và tính trung bình không trọng số giữa các trạm
    combined["timestamp_utc"] = combined["timestamp_utc"].dt.floor("H")
    mean_df = (
        combined
        .groupby(["timestamp_utc", "station_id"])[parameter]
        .mean()           # trung bình trong giờ đó của mỗi trạm
        .reset_index()
        .groupby("timestamp_utc")[parameter]
        .mean()           # trung bình không trọng số giữa các trạm
        .reset_index()
    )
    return mean_df


print("✅ DMI fetch functions sẵn sàng.")

In [ ]:
# DMI không cần API key kể từ 2/12/2025 — tiến hành trực tiếp
print("✅ DMI API mở hoàn toàn (không cần xác thực kể từ 2/12/2025).")
print("   Tuân thủ fair use: sleep giữa các request đã được cấu hình.")

In [ ]:
CACHE_WEATHER = os.path.join(RAW_DIR, "weather.parquet")

if os.path.exists(CACHE_WEATHER):
    print("📂 Đọc weather từ cache...")
    df_weather = pd.read_parquet(CACHE_WEATHER)

else:
    print("🌐 Đang tải dữ liệu thời tiết từ DMI (có thể mất 30–60 phút)...")
    zone_dfs = []

    for zone_name, stations in [("DK1", STATIONS_DK1), ("DK2", STATIONS_DK2)]:
        print(f"\n--- Zone {zone_name} ---")
        param_dfs = []
        for param in DMI_PARAMETERS:
            print(f"  Biến: {param}")
            df_p = fetch_dmi_zone(stations, param, START_DATE, END_DATE, DMI_API_KEY)
            param_dfs.append(df_p)

        # Merge các biến theo timestamp
        df_zone = param_dfs[0].copy()
        for df_p in param_dfs[1:]:
            df_zone = df_zone.merge(df_p, on="timestamp_utc", how="outer")
        df_zone["zone"] = zone_name
        zone_dfs.append(df_zone)

    df_weather = pd.concat(zone_dfs, ignore_index=True)
    df_weather = df_weather.drop_duplicates(subset=["timestamp_utc", "zone"])
    df_weather.to_parquet(CACHE_WEATHER)
    print(f"\n✅ Thời tiết: {len(df_weather):,} hàng")

if not df_weather.empty:
    print(df_weather.head(3))

## 8. Tạo skeleton timestamp đầy đủ (UTC, không có lỗ hổng)

In [ ]:
# Tạo dãy thời gian đầy đủ theo UTC — đây là backbone của dataset
full_range = pd.date_range(
    start=f"{START_DATE} 00:00",
    end=f"{END_DATE} 23:00",
    freq="H",
    tz=TZ_UTC
)

# Cartesian product: mỗi giờ × 2 zone
df_skeleton = pd.DataFrame(
    [(ts, z) for ts in full_range for z in ["DK1", "DK2"]],
    columns=["timestamp_utc", "zone"]
)

print(f"✅ Skeleton: {len(df_skeleton):,} hàng")
print(f"   Kỳ vọng: 3 năm × 365.25 ngày × 24 giờ × 2 zone ≈ {int(3*365.25*24*2):,}")
print(f"   {df_skeleton['timestamp_utc'].min()} → {df_skeleton['timestamp_utc'].max()}")

## 9. Merge tất cả nguồn dữ liệu

In [ ]:
# Đảm bảo tất cả timestamp có timezone UTC
def ensure_utc(df, col="timestamp_utc"):
    s = pd.to_datetime(df[col])
    if s.dt.tz is None:
        return df.assign(**{col: s.dt.tz_localize(TZ_UTC)})
    return df.assign(**{col: s.dt.tz_convert(TZ_UTC)})

df_skeleton   = ensure_utc(df_skeleton)
df_prices     = ensure_utc(df_prices)
df_balance    = ensure_utc(df_balance)
df_cons       = ensure_utc(df_cons)

# Merge dần dần
df = df_skeleton.copy()

df = df.merge(df_prices,  on=["timestamp_utc", "zone"], how="left")
print(f"Sau merge giá:      {len(df):,} hàng")

df = df.merge(df_balance, on=["timestamp_utc", "zone"], how="left")
print(f"Sau merge balance:  {len(df):,} hàng")

df = df.merge(df_cons,    on=["timestamp_utc", "zone"], how="left")
print(f"Sau merge tiêu thụ: {len(df):,} hàng")

if not df_weather.empty:
    df_weather = ensure_utc(df_weather)
    df = df.merge(df_weather, on=["timestamp_utc", "zone"], how="left")
    print(f"Sau merge thời tiết: {len(df):,} hàng")

print(f"\n✅ Merge xong: {len(df):,} hàng × {df.shape[1]} cột")

## 10. Kiểm tra DST — trùng lặp và thiếu hàng

In [ ]:
# Kiểm tra trùng lặp timestamp+zone
dupes = df.duplicated(subset=["timestamp_utc", "zone"]).sum()
print(f"Số hàng trùng lặp (timestamp_utc + zone): {dupes}")
if dupes > 0:
    df = df.drop_duplicates(subset=["timestamp_utc", "zone"], keep="last")
    print(f"   → Đã xóa, còn {len(df):,} hàng")

# Kiểm tra tính liên tục của skeleton
for zone in ["DK1", "DK2"]:
    zone_ts = df.loc[df["zone"] == zone, "timestamp_utc"].sort_values().reset_index(drop=True)
    gaps = zone_ts.diff().dropna()
    expected = pd.Timedelta("1H")
    bad_gaps = gaps[gaps != expected]
    if bad_gaps.empty:
        print(f"{zone}: ✅ Liên tục hoàn toàn (không có lỗ hổng)")
    else:
        print(f"{zone}: ⚠️ {len(bad_gaps)} lỗ hổng thời gian:")
        for idx in bad_gaps.index[:5]:
            print(f"   {zone_ts[idx-1]} → {zone_ts[idx]} (gap={bad_gaps[idx]})")

## 11. Xử lý Missing Data với Forward-fill & Gap Flag

In [ ]:
# Sắp xếp theo zone + thời gian trước khi forward-fill
df = df.sort_values(["zone", "timestamp_utc"]).reset_index(drop=True)

# Cột cần xử lý missing
DATA_COLS = ["spot_price_eur", "imbalance", "mfrr_activated",
             "consumption_mwh", "wind_speed", "radia_glob", "temp_dry"]

# Forward-fill theo từng zone riêng biệt (tránh ffill qua ranh giới zone)
result_parts = []
for zone in ["DK1", "DK2"]:
    zone_df = df[df["zone"] == zone].copy()
    zone_df = forward_fill_with_flag(zone_df, DATA_COLS, max_gap=2)
    result_parts.append(zone_df)

df = pd.concat(result_parts, ignore_index=True)
df = df.sort_values(["timestamp_utc", "zone"]).reset_index(drop=True)

# Báo cáo missing
print("\n=== BÁO CÁO MISSING DATA ===")
for col in DATA_COLS:
    n_null = df[col].isna().sum()
    flag_col = f"{col}_gap_flag"
    n_flagged = df[flag_col].sum() if flag_col in df.columns else 0
    pct = n_null / len(df) * 100
    print(f"  {col:20s}: {n_null:6,} NaN ({pct:.2f}%) | {n_flagged:,} flagged long gaps")

## 12. Thống nhất đơn vị & kiểm tra phạm vi

In [ ]:
# Kiểm tra đơn vị (sanity check)
print("=== KIỂM TRA PHẠM VI GIÁ TRỊ ===")

checks = {
    "spot_price_eur":  (-500, 5000),   # EUR/MWh — âm có thể xảy ra
    "imbalance":       (-5000, 5000),  # MW — có thể âm/dương
    "mfrr_activated":  (0, 3000),      # MW — luôn dương
    "consumption_mwh": (100, 10000),   # MWh — luôn dương và > 0
    "wind_speed":      (0, 50),        # m/s
    "radia_glob":      (0, 1500),      # W/m²
    "temp_dry":        (-40, 45),      # °C
}

for col, (lo, hi) in checks.items():
    if col not in df.columns:
        continue
    valid = df[col].dropna()
    n_out = ((valid < lo) | (valid > hi)).sum()
    print(f"  {col:20s}: min={valid.min():8.2f}, max={valid.max():8.2f}, "
          f"mean={valid.mean():8.2f} | outliers [{lo},{hi}]: {n_out}")

## 13. Chọn cột cuối cùng & Xuất CSV

In [ ]:
# Cột xuất cuối cùng
FINAL_COLS = [
    "timestamp_utc",
    "zone",
    "spot_price_eur",
    "imbalance",
    "mfrr_activated",
    "consumption_mwh",
    "wind_speed",
    "radia_glob",
    "temp_dry",
]

# Thêm các flag columns vào bên cạnh (optional — xóa nếu không cần)
FLAG_COLS = [c for c in df.columns if c.endswith("_gap_flag")]

df_out = df[FINAL_COLS + FLAG_COLS].copy()

# Format timestamp
df_out["timestamp_utc"] = df_out["timestamp_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

OUTPUT_PATH = "denmark_electricity_dataset_2022_2024.csv"
df_out.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Đã xuất: {OUTPUT_PATH}")
print(f"   Kích thước: {len(df_out):,} hàng × {df_out.shape[1]} cột")
print(f"   Dung lượng: {os.path.getsize(OUTPUT_PATH)/1024/1024:.1f} MB")
df_out.head(5)

## 14. Báo cáo tổng kết

In [ ]:
print("=" * 60)
print("          TỔNG KẾT DATASET")
print("=" * 60)

total_rows = len(df_out)
expected   = int(3 * 365.25 * 24 * 2 + 24 * 2)  # ~3 năm + leap days

print(f"Tổng số hàng      : {total_rows:,}")
print(f"Kỳ vọng (~)       : {expected:,}")
print(f"Phạm vi thời gian : {START_DATE} → {END_DATE}")
print(f"Zones             : DK1, DK2")
print()
print("Missing data (sau xử lý):")
for col in FINAL_COLS[2:]:
    # đọc lại từ df_out (đã có dạng string cho timestamp)
    n_null = df[col].isna().sum() if col in df.columns else 0
    pct = n_null / total_rows * 100
    status = "✅" if pct < 1 else ("⚠️" if pct < 5 else "❌")
    print(f"  {status} {col:20s}: {n_null:,} NaN ({pct:.2f}%)")

print()
print(f"File đầu ra: {OUTPUT_PATH}")
print("=" * 60)

---
## Ghi chú & Troubleshooting

### 🌤️ DMI API
Không cần API key hay đăng ký kể từ 2/12/2025. Truy cập thẳng `https://opendataapi.dmi.dk`.
Nếu bị rate limit (HTTP 429): tăng giá trị `time.sleep()` trong `dmi_fetch_station` và `fetch_dmi_zone`.
Tham khảo fair use policy: https://www.dmi.dk/friedata/dokumentation/terms-of-use

### ⚠️ Tên dataset EDS có thể thay đổi
Nếu gặp lỗi `404` hoặc dataset không tìm thấy:
- Kiểm tra tại https://www.energidataservice.dk/
- Tìm dataset **Realtime Electricity Market Data** và cập nhật biến `DATASET_RT`
- Tương tự cho tên cột `IMBALANCE_COL_MAP`

### 🕐 DST handling
- Tháng 3: đồng hồ tiến 1 tiếng (mất 1 hàng — skeleton UTC đảm bảo không bị thiếu)
- Tháng 10: đồng hồ lùi 1 tiếng (`ambiguous='infer'` xử lý trùng lặp)

### 📦 Caching
Dữ liệu thô được lưu trong `./raw_data/*.parquet`. Chạy lại notebook sẽ đọc từ cache, không tải lại.
Xóa file `.parquet` để buộc tải mới.

### 🔢 Đơn vị
| Cột | Đơn vị |
|-----|--------|
| spot_price_eur | EUR/MWh |
| imbalance | MW |
| mfrr_activated | MW |
| consumption_mwh | MWh |
| wind_speed | m/s |
| radia_glob | W/m² |
| temp_dry | °C |